# NB 2.3 &mdash; Solucions dels exercicis

**MP 5134** &mdash; UT2 · *Dades: AEMET, estació de l'aeroport de Palma*

---

Solucionari dels cinc exercicis de la secció 9 del
[NB 2.3](NB_2_3_regressio_polinomica.ipynb) i orientacions per al debat de la
secció 10.

La primera cel·la reconstrueix l'estat del notebook original, amb el test segellat
i les funcions auxiliars. Aquí les funcions reben també la columna objectiu, que
ens farà falta a l'exercici 4.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_absolute_error

URL_DADES = "https://raw.githubusercontent.com/pprohenspolitecnicllevant/disseny-avaluacio-models-ml/refs/heads/main/UT01-Entorn_de_treball_primer_model/aemet/meteo_palma.csv"
df = pd.read_csv(URL_DADES, parse_dates=["fecha"])

# El test segellat (2024-2025) no es toca fins a la UT10
TALL_SEGELLAT = "2024-01-01"
dades = df[df["fecha"] < TALL_SEGELLAT].dropna(subset=["tmed", "tmax"]).copy()
dades["fraccio_any"] = dades["dia_any"] / 366

TALL_VALIDACIO = "2022-01-01"
train = dades[dades["fecha"] < TALL_VALIDACIO]
valid = dades[dades["fecha"] >= TALL_VALIDACIO]


def entrenar_polinomi(grau, dades_entrenament, objectiu="tmed"):
    fabrica = PolynomialFeatures(degree=grau, include_bias=False)
    X = fabrica.fit_transform(dades_entrenament[["fraccio_any"]])
    model = LinearRegression().fit(X, dades_entrenament[objectiu])
    return fabrica, model


def mae_polinomi(fabrica, model, taula, objectiu="tmed"):
    return mean_absolute_error(taula[objectiu], model.predict(fabrica.transform(taula[["fraccio_any"]])))


def escombrada(dades_entrenament, objectiu="tmed"):
    files = []
    for grau in range(1, 16):
        fabrica, model = entrenar_polinomi(grau, dades_entrenament, objectiu)
        files.append({
            "grau": grau,
            "MAE entrenament": mae_polinomi(fabrica, model, dades_entrenament, objectiu),
            "MAE validació": mae_polinomi(fabrica, model, valid, objectiu),
        })
    return pd.DataFrame(files)

print("Estat reconstruït.")

## Exercici 1

> Repeteix l'escombrada amb `random_state` 0, 3 i 7. El millor grau és sempre el 4?

In [ ]:
resum = []
for llavor in [1, 0, 3, 7]:
    poques = train.sample(15, random_state=llavor)
    taula = escombrada(poques)
    millor = taula.loc[taula["MAE validació"].idxmin()]
    resum.append({
        "random_state": llavor,
        "millor grau": int(millor["grau"]),
        "MAE validació del millor": round(millor["MAE validació"], 2),
        "MAE validació grau 15": round(taula.loc[14, "MAE validació"], 0),
    })

pd.DataFrame(resum)

**Resposta.** No. Segons quins quinze dies ens toquin, el millor grau és el **2, el
3 o el 4**, i l'error del millor model va de 2,4 °C a 3,3 °C. El que no canvia mai
és el desastre dels graus alts: el grau 15 dóna errors que van de desenes de graus
fins a centenars de milers.

La conclusió és que **amb poques dades, la tria de l'hiperparàmetre depèn de la
sort**. Si haguéssim de decidir el grau mirant una sola mostra, podríem triar
qualsevol valor entre el 2 i el 4 i convèncer-nos que és *el bo*.

*Per a la correcció:* és la mateixa idea de l'exercici 4 del NB 2.2, ara amb un
hiperparàmetre en lloc d'una mètrica. Val la pena ajuntar les dues observacions
explícitament i anunciar que la **UT7** és la resposta a aquest problema.

## Exercici 2

> Repeteix la secció 6 amb 40 dies en lloc de 15.

In [ ]:
quaranta = train.sample(40, random_state=1)
taula_40 = escombrada(quaranta)

plt.figure(figsize=(8, 4))
plt.plot(taula_40["grau"], taula_40["MAE entrenament"], marker="o", label="entrenament (40 dies)")
plt.plot(taula_40["grau"], taula_40["MAE validació"], marker="o", label="validació")
plt.ylim(0, 10)
plt.xticks(range(1, 16))
plt.xlabel("Grau del polinomi")
plt.ylabel("MAE (°C)")
plt.legend()
plt.show()

taula_40.round(2)

**Resposta.** El mínim de la validació es desplaça al **grau 5**, amb un error
d'uns 2,1 °C, millor que el millor model amb quinze dies. Els graus alts continuen
sobreajustant, però molt menys: el grau 15 dóna uns 29 °C d'error, que és dolent
però lluny dels 72 °C d'abans.

Fixa't també que la validació ja no fa una vall neta: puja i baixa entre el grau 5
i el 9. Amb 40 punts encara hi ha forats on la corba es pot desbocar una mica.

*Per a la correcció:* la resposta que cal buscar és que **amb més dades, el
sobreajust arriba més tard i és menys greu**, i que el grau òptim pot pujar. És la
versió intermèdia de la secció 7 del notebook, entre quinze dies i dos mil cinc-cents.

## Exercici 3

> Amb els quinze dies de la secció 5, prediu la temperatura del 30 d'abril
> (dia 120) amb el grau 4 i amb el grau 15.

In [ ]:
poques = train.sample(15, random_state=1)
trenta_abril = pd.DataFrame({"fraccio_any": [120 / 366]})

for grau in [4, 15]:
    fabrica, model = entrenar_polinomi(grau, poques)
    print(f"Grau {grau:>2}: {model.predict(fabrica.transform(trenta_abril))[0]:6.1f} °C")

print(f"Real, mitjana dels 30 d'abril de 2015-2021: {train.loc[train['dia_any'] == 120, 'tmed'].mean():.1f} °C")
print()
print("Dies de l'any dels quinze punts:", sorted(poques["dia_any"]))

**Resposta.** La temperatura real d'un 30 d'abril és d'uns **16 °C**. El grau 15
prediu **49 graus sota zero**. El grau 4 diu uns 21 °C, que és molt més creïble
però també s'equivoca en cinc graus.

La llista de dies ho explica: entre el **dia 64** (5 de març) i el **dia 171** (20
de juny) no hi ha cap mesura. El 30 d'abril cau al bell mig d'aquest forat.

- El grau 15 no té res que el retingui dins del forat, i la corba s'hi desboca.
- El grau 4 és prou simple per travessar el forat amb una corba suau, però no té
  cap dada de primavera per saber exactament per on ha de passar.

*Per a la correcció:* la segona observació és la més fina i no cal exigir-la. Però
si surt, val la pena remarcar-la: **un model raonable també s'equivoca on no té
dades**. El sobreajust empitjora el problema, però no és l'única causa.

## Exercici 4

> Repeteix la secció 4.1 fent servir `tmax` en lloc de `tmed`.

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
any_sencer = pd.DataFrame({"fraccio_any": np.linspace(0, 1, 366)})

for eix, objectiu in zip(eixos, ["tmed", "tmax"]):
    eix.scatter(train["dia_any"], train[objectiu], alpha=0.1, s=6)
    for grau in [2, 4]:
        fabrica, model = entrenar_polinomi(grau, train, objectiu)
        eix.plot(any_sencer["fraccio_any"] * 366, model.predict(fabrica.transform(any_sencer)),
                 linewidth=2, label=f"grau {grau}: MAE {mae_polinomi(fabrica, model, valid, objectiu):.2f} °C")
    eix.set_title(objectiu)
    eix.set_xlabel("Dia de l'any")
    eix.legend()

eixos[0].set_ylabel("Temperatura (°C)")
plt.show()

**Resposta.** La forma és la mateixa, i el **grau 4** torna a ser suficient. Els
errors de la màxima són una mica més grans que els de la mitjana: uns 2,15 °C
contra 1,97 °C amb el grau 4.

El motiu és que **la temperatura màxima varia més d'un dia per l'altre que la
mitjana**. La màxima depèn molt de coses puntuals, com un dia ennuvolat o una
ratxa de vent; la mitjana, que combina el dia i la nit, les esmorteeix. Com més
soroll té la variable, més gruix té el núvol i més error queda que cap corba pot
eliminar.

*Per a la correcció:* l'exercici també avalua si l'alumnat sap adaptar funcions que
tenen el nom d'una columna *escrit a dins*. Hi ha dues solucions vàlides: copiar i
canviar el nom, o afegir un paràmetre com hem fet aquí. La segona és la que cal
valorar més, i connecta amb el que ja saben de programació.

## Exercici 5

> Explica la diferència entre un model que aprèn i un que memoritza, amb un exemple
> de fora de la informàtica.

**Resposta orientativa**, amb un exemple de cuina:

> Una persona que *aprèn* a cuinar entén per què es fan les coses: que la ceba s'ha
> de sofregir a foc lent, que la sal s'ha d'afegir a poc a poc. Quan li falta un
> ingredient, improvisa i el plat surt bé. Una persona que *memoritza* una recepta
> la reprodueix a la perfecció, però si un dia no té exactament els mateixos
> ingredients o la mateixa cuina, no sap què fer. El model de grau 15 és el segon
> cuiner: reprodueix perfectament els quinze punts que ha vist, i fa disbarats
> amb qualsevol dia que no hi sigui.

Altres exemples que solen sortir i són vàlids: aprendre un idioma contra memoritzar
frases fetes, un conductor que coneix un sol trajecte contra un que sap conduir,
un músic que toca d'oïda contra un que ha après una sola cançó de memòria.

*Per a la correcció:* cal que l'exemple tingui els tres elements: **unes dades
d'entrenament** (la recepta, el trajecte), **una situació nova** i **una diferència
de comportament** davant de la situació nova. Si falta la situació nova, l'exemple
no captura el sobreajust.

## Orientacions per al debat

**R2 de 0,99 sobre l'entrenament.** La primera pregunta ha de ser *"i sobre la
validació?"*. Un número d'entrenament, sol, no diu res de com funcionarà el model.
La segona pregunta bona és *"amb quantes dades?"*, i la tercera, que enllaça amb
la UT1, *"segur que no hi ha la resposta entre les entrades?"*.

**Situacions amb poques dades.** Un producte nou, una malaltia rara, una botiga
acabada d'obrir, una màquina industrial que s'avaria poques vegades. Les
estratègies que convé que surtin: fer servir models simples, aprofitar dades de
casos semblants (productes de la mateixa categoria, altres botigues), esperar a
tenir més dades abans de confiar en el model, o combinar el model amb el criteri
d'una persona experta.

**Aconseguir més dades** no sempre és possible, i gairebé mai és gratis: pot
costar temps (esperar mesos), diners (sensors, enquestes), o tenir limitacions
legals i ètiques (dades mèdiques o personals). Aquest últim punt enllaça amb la
competència transversal CS2 i amb la protecció de dades.